# C_capsule3d — HDC 球壳 X 射线成像仿真 (capsule_test_0809)

## 成像系统 (两段式)
```
X射线段 (rave-sim 仿真)              可见光段 (后处理)
═══════════════════════              ════════════════
源 → 40mm → 样品 → 25mm → 闪烁体     闪烁体 → 20× 物镜 → CCD
                                    
相干衍射 · 波前传播                  非相干强度成像
强度 = |ψ|²                         强度 ×20 放大
```
- **靶丸**: HDC 球壳, Ø900μm, 厚50μm, ρ≈3.5 g/cm³
- **源-物距**: 40 mm (z_start = 0.04 m)
- **物-闪烁体距**: 25 mm (z_detector = 0.065 m)
- **几何放大**: 1.625×
- **总放大**: 20× (可见光显微镜, 后处理)
- **Fresnel 缩放**: 启用 (处理短源距 Nyquist)
  - z_eff = 0.04×0.025/(0.04+0.025) = **15.4 mm**
- **X射线能量**: 10 keV (λ ≈ 0.124 nm)
- **仿真模式**: 2D

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import os

In [ ]:
rave_sim_dir = Path('/mnt/d/rave-sim-main/rave-sim-main')
simulations_dir = Path('/mnt/d/rave-sim-main/rave-sim-main/output')
scratch_dir = simulations_dir

In [ ]:
sys.path.insert(0, str(rave_sim_dir / "big-wave"))
import multisim
import config
import util

In [ ]:
# X射线段参数:
# z_s = z_start - z_source = 0.04 - 0.00 = 0.04 m
# z_d = z_detector - z_start = 0.065 - 0.04 = 0.025 m
# z_eff = (z_s * z_d) / (z_s + z_d) = 0.001 / 0.065 = 0.0154 m = 15.4 mm
# M_geo = (z_s + z_d) / z_s = 0.065 / 0.04 = 1.625×
# 胶囊像@闪烁体: 0.9mm × 1.625 = 1.46mm → detector_size = 2mm (含余量)

config_dict = {
    "sim_params": {
        "is2d": 'true',
        "N": 16384 * 16384,
        "nx": 16384,
        "ny": 16384,
        "dx": 8.5e-8,          # 85 nm, FOV = 1.39 mm
        "z_detector": 0.065,   # 65 mm = 40mm + 25mm (闪烁体位置)
        "detector_size": 2.0e-3,        # 2 mm (覆盖 1.46mm 胶囊像)
        "detector_size_y": 2.0e-3,
        "detector_pixel_size_x": 4e-7,  # 400 nm (有效像素=400/1.625=246nm)
        "detector_pixel_size_y": 4e-7,
        "chunk_size": 2*1024*1024*1024 // 16,
        "use_fresnel_scaling": True,    # 处理短源距 Nyquist
    },
    "use_disk_vector": False,
    "save_final_u_vectors": False,
    "dtype": "c8",
    "multisource": {
        "type": "points",
        "energy_range": [9999, 10001],   # 10 keV
        "x_range": [0, 0],
        "y_range": [0, 0],
        "z": 0.0,                         # source at z=0
        "nr_source_points": 1,
        "seed": 1,
    },
    "elements": [
        {
            "type": "sample",
            "z_start": 0.04,              # 40 mm from source
            "pixel_size_x": 5e-8,         # 50 nm internal sample resolution
            "pixel_size_y": 5e-8,
            "pixel_size_z": 5e-8,
            "grid_path": "/mnt/d/rave-sim-main/rave-sim-main/grid/900um_50um_hdc_shell.npy",
            "materials": [["C", 3.5]],    # HDC ~3.5 g/cm^3
            "x_positions": [0],
            "y_positions": [0],
        },
    ],
}

print("=== X射线段: Fresnel Scaling 参数 ===")
z_s = config_dict["elements"][0]["z_start"] - config_dict["multisource"]["z"]
z_d = config_dict["sim_params"]["z_detector"] - config_dict["elements"][0]["z_start"]
z_eff = (z_s * z_d) / (z_s + z_d)
M_geo = (z_s + z_d) / z_s
print("z_s={:.3f}m, z_d={:.3f}m → z_eff={:.4f}m ({:.1f}mm), M_geo={:.3f}x".format(
    z_s, z_d, z_eff, z_eff*1e3, M_geo))
print("FOV: {:.2f}μm (capsule={:.0f}μm)".format(
    config_dict["sim_params"]["nx"] * config_dict["sim_params"]["dx"] * 1e6, 900))
eff_pixel = config_dict["sim_params"]["detector_pixel_size_x"] / M_geo
nr_pix = int(config_dict["sim_params"]["detector_size"] / config_dict["sim_params"]["detector_pixel_size_x"])
eff_extent = nr_pix * eff_pixel * 1e6
print("Detector: {}×{} px, phys_pixel={}nm, eff_pixel={:.0f}nm, eff_extent={:.0f}μm".format(
    nr_pix, nr_pix,
    config_dict["sim_params"]["detector_pixel_size_x"]*1e9,
    eff_pixel*1e9, eff_extent))
print("可见光20×后: 1像素 = {:.0f}nm @样品面".format(eff_pixel*1e9/20))
print("GPU: {} MB".format(config_dict["sim_params"]["N"] * 8 // (1024*1024)))

In [ ]:
sim_path = multisim.setup_simulation(config_dict, Path("."), simulations_dir)

In [ ]:
computed = config.load(Path(sim_path / 'computed.yaml'))

# print("cutoff angles:", computed['cutoff_angles'])
# print("source points:", computed['source_points'])

In [ ]:
# Run the 2D simulation for each source point
for i in tqdm(range(config_dict["multisource"]["nr_source_points"])):
    os.system(f"CUDA_VISIBLE_DEVICES=0 /mnt/d/rave-sim-main/rave-sim-main/fast-wave/build-Release/fastwave -s {i} {sim_path}")

In [ ]:
wavefronts = util.load_wavefronts_filtered(sim_path, x_range=(-30e-6, 30e-6))
print("nr sources loaded:", len(wavefronts))
wavef = [result[0] for result in wavefronts]
wf = np.sum(wavef, axis=0)
print("nr phase steps:", wf.shape[0])
print("nr detector pixels:", wf.shape[1], wf.shape[2])

In [ ]:
sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
# print(detector_x)

In [ ]:
plt.imshow(wf[0], cmap='inferno')
plt.colorbar()
plt.show()

## History

To see the interference pattern in empty space, we can record slices throughout the simulation and then plot them. `run_single_simulation` takes an optional argument `history_dz` defining the resolution with which we record the history.

Note that the history is not necessarily recorded with a constant z-spacing. Inside gratings and samples, one slice is recorded for every step. The history also records a list of z-coordinates at which the slices were recorded, which we can use for plotting.

In [ ]:
multisim.run_single_simulation(sim_path, 1, scratch_dir, save_keypoints_path=None, history_dz=0.02)

In [ ]:
# Path to the directory for the source with index 1
source_dir = multisim.get_sub_dir(sim_path, 1)

hist_x = np.load(source_dir / "history_x.npy")
hist_z = np.load(source_dir / "history_z.npy")
hist = np.load(source_dir / "history.npy")
plt.pcolormesh(
    hist_z,
    hist_x,
    hist,
    cmap="Greys_r",
    vmin=0,
    vmax=1e-6,
    shading="nearest",
)
plt.xlabel("z (m)")
plt.ylabel("x (m)")